In [2]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings


In [2]:


# Load the engineered dataset
df_engineered = pd.read_parquet('results/df_engineered.parquet')

print(f"\nDataset shape: {df_engineered.shape}")



Dataset shape: (204942, 79)


In [3]:
df_engineered.head()

,timestamp,ping_ms,datarate,jitter,Latitude,Longitude,Altitude,speed_kmh,COG,precipIntensity,...,PCell_freq_MHz_missing,PCell_E-ARFCN_missing,device_pc1,device_pc2,device_pc3,device_pc4,direction_uplink,measured_qos_delay,hour,day_of_week
216,2021-06-22 09:49:54+02:00,NaN,68700000.0,0.000848,52.514013,13.335172,41.9,0.0,0.0,0.0652,...,0,0,1,0,0,0,0,0,9,1
217,2021-06-22 09:49:54+02:00,NaN,24500000.0,0.000512,52.514008,13.335195,35.3,0.0,259.0,0.0652,...,0,0,0,1,0,0,0,0,9,1
218,2021-06-22 09:49:54+02:00,NaN,49800000.0,0.000090,52.513830,13.334935,30.7,0.0,0.0,0.0653,...,0,0,0,0,1,0,0,0,9,1
219,2021-06-22 09:49:54+02:00,1396.0,70500000.0,0.000207,52.513848,13.334832,32.3,0.0,265.9,0.0653,...,0,0,0,0,0,1,0,0,9,1
220,2021-06-22 09:49:55+02:00,NaN,20800000.0,0.002268,52.514005,13.335195,35.4,0.0,259.0,0.0652,...,0,0,0,1,0,0,0,0,9,1


In [24]:

print("TEMPORAL SPLITTING STRATEGY")

# Check data distribution by date
df_engineered['date'] = df_engineered['timestamp'].dt.date

print("Data distribution by date:")
date_counts = df_engineered.groupby('date').size()
for date, count in date_counts.items():
    pct = (count / len(df_engineered)) * 100
    print(f"  {date}: {count:>7,} rows ({pct:>5.2f}%)")

print(f"\nTotal: {len(df_engineered):,} rows")

TEMPORAL SPLITTING STRATEGY
Data distribution by date:
  2021-06-22:  79,564 rows (38.82%)
  2021-06-23:  79,015 rows (38.55%)
  2021-06-24:  46,363 rows (22.62%)

Total: 204,942 rows


In [25]:

print("EXECUTING TEMPORAL SPLIT (70/15/15)")

# Sort by timestamp to ensure chronological order
df_sorted = df_engineered.sort_values('timestamp').reset_index(drop=True)

print(f"\nSorted by timestamp")
# Calculate split indices
n_total = len(df_sorted)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
n_test = n_total - n_train - n_val  # Remainder goes to test

print(f"\n SPLIT SIZES:")
print(f"  Total:      {n_total:>7,} rows (100.0%)")
print(f"  Train:      {n_train:>7,} rows ({n_train/n_total*100:>5.1f}%)")
print(f"  Validation: {n_val:>7,} rows ({n_val/n_total*100:>5.1f}%)")
print(f"  Test:       {n_test:>7,} rows ({n_test/n_total*100:>5.1f}%)")

# Split the data
train_data = df_sorted.iloc[:n_train].copy()
val_data = df_sorted.iloc[n_train:n_train+n_val].copy()
test_data = df_sorted.iloc[n_train+n_val:].copy()

print(f"\n Data split complete")

# Verify temporal order (no leakage)
print(f"\nTEMPORAL VERIFICATION (No leakage check):")
print(f"  Train period:      {train_data['timestamp'].min()} to {train_data['timestamp'].max()}")
print(f"  Validation period: {val_data['timestamp'].min()} to {val_data['timestamp'].max()}")
print(f"  Test period:       {test_data['timestamp'].min()} to {test_data['timestamp'].max()}")

# Check no overlap
train_max = train_data['timestamp'].max()
val_min = val_data['timestamp'].min()
val_max = val_data['timestamp'].max()
test_min = test_data['timestamp'].min()

if train_max < val_min and val_max < test_min:
    print(f"\n Train < Val < Test")
else:
    print(f"\n Possible temporal overlap!")


EXECUTING TEMPORAL SPLIT (70/15/15)

Sorted by timestamp

 SPLIT SIZES:
  Total:      204,942 rows (100.0%)
  Train:      143,459 rows ( 70.0%)
  Validation:  30,741 rows ( 15.0%)
  Test:        30,742 rows ( 15.0%)

 Data split complete

TEMPORAL VERIFICATION (No leakage check):
  Train period:      2021-06-22 09:49:54+02:00 to 2021-06-23 15:51:37+02:00
  Validation period: 2021-06-23 15:51:37+02:00 to 2021-06-24 10:19:02+02:00
  Test period:       2021-06-24 10:19:02+02:00 to 2021-06-24 18:59:59+02:00

 Possible temporal overlap!


In [ ]:

# Move overlapping train-val timestamp to validation
overlap_timestamp_1 = train_data['timestamp'].max()
if overlap_timestamp_1 in val_data['timestamp'].values:
    mask = train_data['timestamp'] == overlap_timestamp_1
    rows_to_move = train_data[mask].copy()
    train_data = train_data[~mask].reset_index(drop=True)
    val_data = pd.concat([rows_to_move, val_data]).sort_values('timestamp').reset_index(drop=True)
    print(f"  Moved {len(rows_to_move)} rows from train to validation")

# Move overlapping val-test timestamp to test
overlap_timestamp_2 = val_data['timestamp'].max()
if overlap_timestamp_2 in test_data['timestamp'].values:
    mask = val_data['timestamp'] == overlap_timestamp_2
    rows_to_move = val_data[mask].copy()
    val_data = val_data[~mask].reset_index(drop=True)
    test_data = pd.concat([rows_to_move, test_data]).sort_values('timestamp').reset_index(drop=True)
    print(f"  Moved {len(rows_to_move)} rows from validation to test")

# Verify no overlap now
train_timestamps = set(train_data['timestamp'])
val_timestamps = set(val_data['timestamp'])
test_timestamps = set(test_data['timestamp'])

overlap_check = (
    len(train_timestamps & val_timestamps) + 
    len(val_timestamps & test_timestamps) + 
    len(train_timestamps & test_timestamps)
)

if overlap_check == 0:
    print(f"\n  NO OVERLAP - Clean split achieved!")
else:
    print(f"\n Still {overlap_check} overlapping timestamps")

# Final split sizes
print(f"\n FINAL SPLIT SIZES:")
print(f"  Train:      {len(train_data):>7,} rows ({len(train_data)/len(df_sorted)*100:>5.1f}%)")
print(f"  Validation: {len(val_data):>7,} rows ({len(val_data)/len(df_sorted)*100:>5.1f}%)")
print(f"  Test:       {len(test_data):>7,} rows ({len(test_data)/len(df_sorted)*100:>5.1f}%)")
print(f"  Total:      {len(train_data)+len(val_data)+len(test_data):>7,} rows")


train_data.to_pickle('dataset/train_data.pkl')
val_data.to_pickle('dataset/val_data.pkl')
test_data.to_pickle('dataset/test_data.pkl')

print("Saved: dataset/train_data.pkl")
print("Saved: dataset/val_data.pkl")
print("Saved: dataset/test_data.pkl")

# Save split indices for reference
split_info = {
    'n_train': len(train_data),
    'n_val': len(val_data),
    'n_test': len(test_data),
    'train_start': str(train_data['timestamp'].min()),
    'train_end': str(train_data['timestamp'].max()),
    'val_start': str(val_data['timestamp'].min()),
    'val_end': str(val_data['timestamp'].max()),
    'test_start': str(test_data['timestamp'].min()),
    'test_end': str(test_data['timestamp'].max()),
    'split_date': str(datetime.now()),
    'random_seed': 42
}

import json
with open('dataset/split_info_1.json', 'w') as f:
    json.dump(split_info, f, indent=2)

print("Saved: dataset/split_info.json")




 Still 2 overlapping timestamps

 FINAL SPLIT SIZES:
  Train:      143,459 rows ( 70.0%)
  Validation:  30,741 rows ( 15.0%)
  Test:        30,742 rows ( 15.0%)
  Total:      204,942 rows
Saved: dataset/train_data.pkl
Saved: dataset/val_data.pkl
Saved: dataset/test_data.pkl
Saved: dataset/split_info.json


In [3]:
train_data = pd.read_pickle('dataset/train_data.pkl')
val_data = pd.read_pickle('dataset/val_data.pkl')
test_data = pd.read_pickle('dataset/test_data.pkl')

### Artificial Masking

In [4]:

# Set random seed for reproducibility
np.random.seed(42)

# Mask rate
MASK_RATE = 0.15

print(f"\n  CONFIGURATION:")
print(f"    Mask rate: {MASK_RATE*100:.0f}%")
print(f"    Random seed: 42")
print(f"    Target sets: Validation & Test only")

# Identify numeric columns to mask (exclude metadata, identifiers, engineered features)
exclude_cols = ['timestamp', 'ts_gps', 'device', 
                'scenario', 'direction', 'measured_qos', 'operator',
                'date', 'hour', 'day_of_week',]

# Also exclude missing indicator columns and one-hot encoded columns
exclude_cols += [col for col in val_data.columns if col.endswith('_missing')]
exclude_cols += [col for col in val_data.columns if any(prefix in col for prefix in 
                ['device_', 'scenario_', 'drive_mode_', 'direction_', 'measured_qos_', 'area_','PCell_Downlink_frequency_','PCell_freq_MHz_'])]

# Get columns to mask (numeric features only)
all_numeric = val_data.select_dtypes(include=[np.number]).columns.tolist()
cols_to_mask = [col for col in all_numeric if col not in exclude_cols]




  CONFIGURATION:
    Mask rate: 15%
    Random seed: 42
    Target sets: Validation & Test only


In [5]:

print("EXECUTING MASKING ON VALIDATION & TEST SETS")

# Function to create masks
def create_evaluation_masks(data, cols_to_mask, mask_rate=0.15):
    
    masked_data = data.copy()
    ground_truth = data[cols_to_mask].copy()
    mask_indices = pd.DataFrame(False, index=data.index, columns=cols_to_mask)
    
    total_masked = 0
    
    for col in cols_to_mask:
        # Find non-missing values
        non_missing_mask = data[col].notna()
        non_missing_indices = data[non_missing_mask].index
        n_non_missing = len(non_missing_indices)
        
        if n_non_missing == 0:
            continue  
        
        # Randomly select mask_rate% to mask
        n_to_mask = int(n_non_missing * mask_rate)
        
        if n_to_mask > 0:
            indices_to_mask = np.random.choice(non_missing_indices, 
                                              size=n_to_mask, 
                                              replace=False)
            
            # Apply mask
            masked_data.loc[indices_to_mask, col] = np.nan
            mask_indices.loc[indices_to_mask, col] = True
            
            total_masked += n_to_mask
    
    return masked_data, ground_truth, mask_indices, total_masked




EXECUTING MASKING ON VALIDATION & TEST SETS


In [7]:
print("\n MASKING VALIDATION SET...")

val_masked, val_ground_truth, val_mask_indices, val_total_masked = create_evaluation_masks(
    val_data, cols_to_mask, MASK_RATE
)

val_total_cells = len(val_data) * len(cols_to_mask)
val_originally_missing = val_data[cols_to_mask].isnull().sum().sum()
val_non_missing = val_total_cells - val_originally_missing
val_masked_pct = (val_total_masked / val_non_missing * 100)

print(f"  Total cells: {val_total_cells:,}")
print(f"  Originally missing: {val_originally_missing:,}")
print(f"  Non-missing cells: {val_non_missing:,}")
print(f"  Artificially masked: {val_total_masked:,} ({val_masked_pct:.2f}% of non-missing)")


print("\n MASKING TEST SET...")

test_masked, test_ground_truth, test_mask_indices, test_total_masked = create_evaluation_masks(
    test_data, cols_to_mask, MASK_RATE
)

test_total_cells = len(test_data) * len(cols_to_mask)
test_originally_missing = test_data[cols_to_mask].isnull().sum().sum()
test_non_missing = test_total_cells - test_originally_missing
test_masked_pct = (test_total_masked / test_non_missing * 100)

print(f"  Total cells: {test_total_cells:,}")
print(f"  Originally missing: {test_originally_missing:,}")
print(f"  Non-missing cells: {test_non_missing:,}")
print(f"  Artificially masked: {test_total_masked:,} ({test_masked_pct:.2f}% of non-missing)")

print(f"\nValidation set:")
print(f"    Before masking: {(val_data[cols_to_mask].isnull().sum().sum() / val_total_cells * 100):.2f}% missing")
print(f"    After masking:  {(val_masked[cols_to_mask].isnull().sum().sum() / val_total_cells * 100):.2f}% missing")

print(f"\nTest set:")
print(f"    Before masking: {(test_data[cols_to_mask].isnull().sum().sum() / test_total_cells * 100):.2f}% missing")
print(f"    After masking:  {(test_masked[cols_to_mask].isnull().sum().sum() / test_total_cells * 100):.2f}% missing")



 MASKING VALIDATION SET...
  Total cells: 1,137,417
  Originally missing: 12,524
  Non-missing cells: 1,124,893
  Artificially masked: 168,723 (15.00% of non-missing)

 MASKING TEST SET...
  Total cells: 1,137,454
  Originally missing: 21,516
  Non-missing cells: 1,115,938
  Artificially masked: 167,377 (15.00% of non-missing)

Validation set:
    Before masking: 1.10% missing
    After masking:  15.93% missing

Test set:
    Before masking: 1.89% missing
    After masking:  16.61% missing


In [30]:

print("SAVING MASKED DATA & GROUND TRUTH")

# Save train data (no masking - use as-is)
train_data.to_pickle('dataset/train_data_final.pkl')
print(" Saved: dataset/train_data_final.pkl (original, no masking)")

# Save validation - masked version
val_masked.to_pickle('dataset/val_data_masked.pkl')
print(" Saved: dataset/val_data_masked.pkl (with artificial masks)")

# Save validation - ground truth
val_ground_truth.to_pickle('dataset/val_ground_truth.pkl')
print(" Saved: dataset/val_ground_truth.pkl (original values)")

# Save validation - mask indices
val_mask_indices.to_pickle('dataset/val_mask_indices.pkl')
print(" Saved: dataset/val_mask_indices.pkl (boolean mask)")

# Save test - masked version
test_masked.to_pickle('dataset/test_data_masked.pkl')
print(" Saved: dataset/test_data_masked.pkl (with artificial masks)")

# Save test - ground truth
test_ground_truth.to_pickle('dataset/test_ground_truth.pkl')
print(" Saved: dataset/test_ground_truth.pkl (original values)")

# Save test - mask indices
test_mask_indices.to_pickle('dataset/test_mask_indices.pkl')
print(" Saved: dataset/test_mask_indices.pkl (boolean mask)")

# Save list of masked columns
with open('dataset/cols_to_mask.json', 'w') as f:
    json.dump(cols_to_mask, f, indent=2)
print(" Saved: dataset/cols_to_mask.json (list of masked columns)")

# Create summary
summary = {
    'train': {
        'n_rows': len(train_data),
        'n_cols': len(train_data.columns),
        'masking': 'none',
        'file': 'train_data_final.pkl'
    },
    'validation': {
        'n_rows': len(val_masked),
        'n_cols': len(val_masked.columns),
        'n_masked_cols': len(cols_to_mask),
        'n_artificially_masked': val_total_masked,
        'mask_rate': f'{MASK_RATE*100:.0f}%',
        'files': {
            'data': 'val_data_masked.pkl',
            'ground_truth': 'val_ground_truth.pkl',
            'mask_indices': 'val_mask_indices.pkl'
        }
    },
    'test': {
        'n_rows': len(test_masked),
        'n_cols': len(test_masked.columns),
        'n_masked_cols': len(cols_to_mask),
        'n_artificially_masked': test_total_masked,
        'mask_rate': f'{MASK_RATE*100:.0f}%',
        'files': {
            'data': 'test_data_masked.pkl',
            'ground_truth': 'test_ground_truth.pkl',
            'mask_indices': 'test_mask_indices.pkl'
        }
    },
    'created_date': str(datetime.now()),
    'random_seed': 42
}

with open('dataset/masking_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(" Saved: dataset/masking_summary.json (summary)")





SAVING MASKED DATA & GROUND TRUTH
 Saved: dataset/train_data_final.pkl (original, no masking)
 Saved: dataset/val_data_masked.pkl (with artificial masks)
 Saved: dataset/val_ground_truth.pkl (original values)
 Saved: dataset/val_mask_indices.pkl (boolean mask)
 Saved: dataset/test_data_masked.pkl (with artificial masks)
 Saved: dataset/test_ground_truth.pkl (original values)
 Saved: dataset/test_mask_indices.pkl (boolean mask)
 Saved: dataset/cols_to_mask.json (list of masked columns)
 Saved: dataset/masking_summary.json (summary)
